In [ ]:
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

GAME = "havannah"
OPEN_SPIEL_GAME = "havannah(board_size=7)"
RUN_VARIANT = "agentic"
LLM_MODEL = "openai-codex/gpt-5.5:xhigh"
TIMEOUT_SECONDS = 1800 if RUN_VARIANT == "agentic" else 900

USE_OPEN_SPIEL_BACKBONE = True
USE_IMPLEMENTATION_BRIEF = True

ROLLOUTS = 1000
MAX_STEPS = 1000
CHECK_SEED = 1

OUTPUT_DIR = Path("outputs")
PROMPT_PATH = Path("prompts/rulebook_to_python.txt")
BACKBONE_PATH = Path("prompts/open_spiel_backbone.md")
LLM_JUDGE_PROMPT_PATH = Path("prompts/llm_judge_review.md")
IMPLEMENTATION_BRIEF_PATH = OUTPUT_DIR / f"{GAME}_implementation_brief.md"

VARIANT_STEMS = {
    "oneshot": f"{GAME}_oneshot",
    "agentic": f"{GAME}_agentic",
}
if RUN_VARIANT not in VARIANT_STEMS:
    raise ValueError(f"Unsupported RUN_VARIANT: {RUN_VARIANT}")

RUN_STEM = VARIANT_STEMS[RUN_VARIANT]
CODE_PATH = OUTPUT_DIR / f"{RUN_STEM}.py"
RESPONSE_PATH = OUTPUT_DIR / f"{RUN_STEM}.md"
CHECK_LOG_PATH = OUTPUT_DIR / f"{RUN_STEM}_checks.txt"
JUDGE_PACKET_PATH = OUTPUT_DIR / f"{RUN_STEM}_judge_packet.md"
JUDGE_REVIEW_PATH = OUTPUT_DIR / f"{RUN_STEM}_judge.md"


def variant_paths(variant: str) -> dict[str, Path | str]:
    if variant not in VARIANT_STEMS:
        raise ValueError(f"Unknown variant: {variant}")
    stem = VARIANT_STEMS[variant]
    return {
        "variant": variant,
        "stem": stem,
        "code": OUTPUT_DIR / f"{stem}.py",
        "response": OUTPUT_DIR / f"{stem}.md",
        "check_log": OUTPUT_DIR / f"{stem}_checks.txt",
        "judge_packet": OUTPUT_DIR / f"{stem}_judge_packet.md",
        "judge_review": OUTPUT_DIR / f"{stem}_judge.md",
    }


def find_rules_path() -> Path:
    input_dir = Path("inputs")
    if not input_dir.exists():
        raise FileNotFoundError("Missing inputs/ directory")

    rules_paths = sorted(
        path
        for path in input_dir.iterdir()
        if path.stem == "game_rules" and path.suffix.lower() in {".txt", ".pdf"}
    )
    if not rules_paths:
        raise FileNotFoundError("Keep exactly one of inputs/game_rules.txt or inputs/game_rules.pdf")
    if len(rules_paths) > 1:
        names = ", ".join(path.name for path in rules_paths)
        raise RuntimeError(f"Multiple game_rules files found; keep exactly one: {names}")
    return rules_paths[0]



def read_rules_text(rules_path: Path) -> str:
    if rules_path.suffix.lower() == ".txt":
        return rules_path.read_text(encoding="utf-8")

    if rules_path.suffix.lower() == ".pdf":
        try:
            from pypdf import PdfReader
        except ImportError as exc:
            raise ImportError("PDF rulebooks require pypdf; install requirements.txt in the boardbench Conda env") from exc

        reader = PdfReader(str(rules_path))
        return "\n\n".join(
            page_text.strip()
            for page in reader.pages
            for page_text in [page.extract_text() or ""]
            if page_text.strip()
        )

    raise ValueError(f"Unsupported rules file type: {rules_path.suffix}")


def get_pdf_renderer_path() -> str:
    renderer = shutil.which("pdftoppm") or shutil.which("pdftoppm.exe")
    if renderer:
        return renderer

    windows_fallback = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64/pdftoppm.exe"
    if windows_fallback.exists():
        return str(windows_fallback)

    raise FileNotFoundError(
        "Scanned/image PDFs require pdftoppm to render pages for pi image attachments"
    )


def render_pdf_pages(rules_path: Path) -> list[Path]:
    page_dir = OUTPUT_DIR / "rulebook_pages" / rules_path.stem
    page_dir.mkdir(parents=True, exist_ok=True)
    for old_page in page_dir.glob("page-*.png"):
        old_page.unlink()

    prefix = page_dir / "page"
    subprocess.run(
        [get_pdf_renderer_path(), "-png", "-r", "180", str(rules_path), str(prefix)],
        check=True,
        capture_output=True,
        text=True,
    )
    pages = sorted(page_dir.glob("page-*.png"))
    if not pages:
        raise RuntimeError(f"No PDF page images were rendered from {rules_path}")
    return pages


def build_rules_context() -> tuple[str, list[Path]]:
    rules_path = find_rules_path()
    rules_text = read_rules_text(rules_path)
    if rules_text.strip():
        return "Hier folgt die Spielanleitung:\n\n" + rules_text, []

    if rules_path.suffix.lower() == ".pdf":
        rendered_pages = render_pdf_pages(rules_path)
        page_list = "\n".join(f"- {path.as_posix()}" for path in rendered_pages)
        note = (
            "Die Spielanleitung ist eine bildbasierte PDF ohne extrahierbaren Text. "
            "Die gerenderten PDF-Seiten sind als Datei-Anhänge an diesen pi-Aufruf angehängt. "
            "Verwende ausschließlich diese Seiten als Regelquelle.\n\n"
            f"Gerenderte Seiten:\n{page_list}"
        )
        return note, rendered_pages

    raise ValueError(f"No usable rule text found in {rules_path}")

def optional_generation_inputs() -> list[tuple[str, Path, str]]:
    items: list[tuple[str, Path, str]] = []
    if USE_OPEN_SPIEL_BACKBONE and BACKBONE_PATH.exists():
        items.append(("OpenSpiel backbone", BACKBONE_PATH, BACKBONE_PATH.read_text(encoding="utf-8")))
    if USE_IMPLEMENTATION_BRIEF and IMPLEMENTATION_BRIEF_PATH.exists():
        items.append(
            (
                "Implementation brief",
                IMPLEMENTATION_BRIEF_PATH,
                IMPLEMENTATION_BRIEF_PATH.read_text(encoding="utf-8"),
            )
        )
    return items


def get_pi_path() -> str:
    pi_path = shutil.which("pi") or shutil.which("pi.cmd")
    if pi_path is not None:
        return pi_path

    windows_fallback = Path.home() / "AppData/Roaming/npm/pi.cmd"
    if windows_fallback.exists():
        return str(windows_fallback)

    raise FileNotFoundError("Could not find pi or pi.cmd")


def extract_code_block(text: str) -> str | None:
    match = re.search(r"```python\s*(.*?)```", text, re.IGNORECASE | re.DOTALL)
    if match is None:
        return None
    return match.group(1).strip() + "\n"



def build_one_shot_prompt() -> tuple[str, list[Path]]:
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")

    rules_context, attachments = build_rules_context()
    prompt_parts = [PROMPT_PATH.read_text(encoding="utf-8")]
    for label, path, text in optional_generation_inputs():
        prompt_parts.append(f"# {label}: {path.as_posix()}\n\n{text}")
    return "\n\n".join(prompt_parts) + "\n\n" + rules_context, attachments




def build_agentic_prompt() -> tuple[str, list[Path]]:
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")

    # Build attachments for image-only PDFs, but do not inline hidden check files.
    _rules_context, attachments = build_rules_context()
    lines = [
        "Work in the isolated temporary workspace prepared for this run.",
        "The only source-material directory is inputs/.",
        "Use only files in inputs/ as task context:",
        "- inputs/rulebook_to_python.txt",
        "- inputs/open_spiel_backbone.md, if present",
        "- inputs/implementation_brief.md, if present",
        "- inputs/game_rules.txt or inputs/game_rules.pdf",
        "- inputs/game_rules_extracted.txt, if present",
        "- inputs/rulebook_pages/*.png, if present or attached",
        "",
        "Do not read, copy, infer from, or run BoardBench benchmark checks.",
        "Do not access parent directories or repository files outside this isolated workspace.",
        "The evaluation checks must remain independent and unseen during generation.",
        "",
        "You may write the generated Python module under outputs/, inspect your own generated file,",
        "and run small self-contained Python syntax/import/logical smoke checks against that file.",
        "Do not use outside game knowledge or remembered rules.",
        "Use only the rulebook text or attached/rendered PDF page images as the game source of truth.",
        f"Write the final Python module to {CODE_PATH.as_posix()}.",
        "",
        "Your final response must contain:",
        "1. Open questions / assumptions",
        "2. one fenced python code block with the exact final file content",
    ]
    return "\n".join(lines), attachments


def copy_if_exists(source: Path, target: Path) -> bool:
    if not source.exists():
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    return True


def prepare_agentic_workspace(file_args: list[Path]) -> tuple[Path, list[Path]]:
    """Create a temporary generation workspace without BoardBench checks.

    The generator sees only copied source material in inputs/ plus its own
    outputs/ directory. This prevents leakage from benchmark checks while still
    allowing agentic self-review and syntax/logical smoke checks on its own code.
    """

    workspace = Path(tempfile.mkdtemp(prefix=f"boardbench_{GAME}_agentic_"))
    input_dir = workspace / "inputs"
    output_dir = workspace / "outputs"
    input_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    copy_if_exists(PROMPT_PATH, input_dir / "rulebook_to_python.txt")
    if USE_OPEN_SPIEL_BACKBONE:
        copy_if_exists(BACKBONE_PATH, input_dir / "open_spiel_backbone.md")
    if USE_IMPLEMENTATION_BRIEF:
        copy_if_exists(IMPLEMENTATION_BRIEF_PATH, input_dir / "implementation_brief.md")

    rules_path = find_rules_path()
    copy_if_exists(rules_path, input_dir / rules_path.name)
    rules_text = read_rules_text(rules_path)
    if rules_text.strip():
        (input_dir / "game_rules_extracted.txt").write_text(rules_text, encoding="utf-8")

    copied_args: list[Path] = []
    if file_args:
        page_dir = input_dir / "rulebook_pages"
        page_dir.mkdir(parents=True, exist_ok=True)
        for path in file_args:
            target = page_dir / path.name
            shutil.copy2(path, target)
            copied_args.append(target)

    return workspace, copied_args


def rewrite_attachment_paths(prompt_text: str, original_args: list[Path], copied_args: list[Path]) -> str:
    for original, copied in zip(original_args, copied_args):
        prompt_text = prompt_text.replace(original.as_posix(), copied.as_posix())
    return prompt_text


def copy_workspace_code(workspace: Path) -> bool:
    workspace_code_path = workspace / CODE_PATH
    if not workspace_code_path.exists():
        return False
    CODE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CODE_PATH.write_text(workspace_code_path.read_text(encoding="utf-8"), encoding="utf-8")
    return True

def pi_command(file_args: list[Path] | None = None) -> list[str]:
    base = [
        get_pi_path(),
        "-p",
        "--no-session",
        "--model",
        LLM_MODEL,
        "--no-extensions",
        "--no-skills",
        "--no-prompt-templates",
        "--no-context-files",
    ]
    if RUN_VARIANT == "oneshot":
        base.append("--no-tools")
    else:
        base += ["--tools", "read,write,edit,bash,grep,find,ls"]

    for path in file_args or []:
        base.append("@" + path.as_posix())
    return base



def run_generation() -> subprocess.CompletedProcess[str]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prompt_text, file_args = build_one_shot_prompt() if RUN_VARIANT == "oneshot" else build_agentic_prompt()

    workspace: Path | None = None
    command_file_args = file_args
    cwd: Path | None = None
    if RUN_VARIANT == "agentic":
        workspace, command_file_args = prepare_agentic_workspace(file_args)
        prompt_text = rewrite_attachment_paths(prompt_text, file_args, command_file_args)
        cwd = workspace

    command = pi_command(command_file_args)

    print(f"variant: {RUN_VARIANT}")
    print(f"game: {GAME}")
    print(f"openspiel game: {OPEN_SPIEL_GAME}")
    print(f"model: {LLM_MODEL}")
    print(f"rules: {find_rules_path()}")
    if cwd is not None:
        print(f"generation workspace: {cwd}")
    if command_file_args:
        print("attached files:")
        for path in command_file_args:
            print(f"- {path}")
    print("command:", shlex.join(command))

    try:
        result = subprocess.run(
            command,
            input=prompt_text,
            capture_output=True,
            text=True,
            timeout=TIMEOUT_SECONDS,
            cwd=str(cwd) if cwd is not None else None,
        )

        RESPONSE_PATH.write_text(result.stdout or "", encoding="utf-8")
        if result.returncode != 0:
            stderr = (result.stderr or "").strip()
            stdout = (result.stdout or "").strip()
            raise RuntimeError(stderr or stdout or "pi call failed")

        code = extract_code_block(result.stdout or "")
        if code is not None:
            CODE_PATH.write_text(code, encoding="utf-8")
            print(f"Saved raw response: {RESPONSE_PATH}")
            print(f"Saved extracted code: {CODE_PATH}")
        elif workspace is not None and copy_workspace_code(workspace):
            print(f"Saved raw response: {RESPONSE_PATH}")
            print(f"Copied code from isolated workspace: {CODE_PATH}")
        elif CODE_PATH.exists():
            print(f"Saved raw response: {RESPONSE_PATH}")
            print(f"No fenced python block found; keeping existing file: {CODE_PATH}")
        else:
            raise RuntimeError("No fenced python block found in the LLM response and no code file was written")

        if (result.stderr or "").strip():
            print("\npi stderr:\n" + result.stderr.strip())
        return result
    finally:
        if workspace is not None:
            shutil.rmtree(workspace, ignore_errors=True)

def check_command(
    code_path: Path,
    judge_path: Path,
    *,
    include_judge: bool = False,
    include_final: bool = False,
) -> list[str]:
    check_script = Path("checks/run_checks.py")
    if not check_script.exists():
        raise FileNotFoundError("Could not find checks/run_checks.py")

    cmd = [
        sys.executable,
        "-u",
        str(check_script),
        "--game",
        OPEN_SPIEL_GAME if include_final else GAME,
        "--code-path",
        str(code_path),
        "--rollouts",
        str(ROLLOUTS),
        "--max-steps",
        str(MAX_STEPS),
        "--seed",
        str(CHECK_SEED),
    ]
    if include_judge:
        cmd.append("--include-judge")
        cmd += ["--judge-path", str(judge_path)]
    if include_final:
        cmd.append("--include-final")
    return cmd


def run_checks_for(
    *,
    code_path: Path,
    judge_path: Path,
    include_judge: bool = False,
    include_final: bool = False,
    log_path: Path | None = None,
    label: str | None = None,
) -> subprocess.CompletedProcess[str]:
    cmd = check_command(
        code_path=code_path,
        judge_path=judge_path,
        include_judge=include_judge,
        include_final=include_final,
    )
    tag = label or code_path.stem
    print(f"running checks for {tag}: {code_path}")
    print("command:", shlex.join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")

    if log_path is not None:
        log_path.write_text(output, encoding="utf-8")
        print(f"Saved check log: {log_path}")

    print(output)
    if result.returncode != 0:
        print(f"checks failed with exit code {result.returncode}")
    return result



def run_pair_action_compare(
    *,
    left_code_path: Path,
    right_code_path: Path,
    left_label: str = "oneshot",
    right_label: str = "agentic",
) -> subprocess.CompletedProcess[str]:
    compare_script = Path("checks/compare_pair.py")
    if not compare_script.exists():
        raise FileNotFoundError("Could not find checks/compare_pair.py")

    cmd = [
        sys.executable,
        "-u",
        str(compare_script),
        "--game",
        GAME,
        "--left-code-path",
        str(left_code_path),
        "--right-code-path",
        str(right_code_path),
        "--left-label",
        left_label,
        "--right-label",
        right_label,
        "--rollouts",
        str(ROLLOUTS),
        "--max-steps",
        str(MAX_STEPS),
        "--seed",
        str(CHECK_SEED),
    ]
    print('\n' + "=" * 80)
    print("running pair action-language comparison")
    print("command:", shlex.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    pair_log_path = OUTPUT_DIR / f"{GAME}_pair_action_compare.txt"
    pair_log_path.write_text(output, encoding="utf-8")
    print(f"Saved pair comparison log: {pair_log_path}")
    print(output)
    if result.returncode != 0:
        print(f"pair action-language comparison failed with exit code {result.returncode}")
    return result



def build_judge_packet_for(
    *,
    code_path: Path,
    check_log_path: Path,
    output_path: Path,
    judge_review_path: Path,
    variant: str,
) -> Path:
    if not code_path.exists():
        raise FileNotFoundError(f"Missing generated code: {code_path}")
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")
    if not LLM_JUDGE_PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing judge prompt file: {LLM_JUDGE_PROMPT_PATH}")

    rules_path = find_rules_path()
    sections = [
        "# BoardBench judge packet",
        f"- game: {GAME}",
        f"- OpenSpiel reference: {OPEN_SPIEL_GAME}",
        f"- variant: {variant}",
        f"- generated code: {code_path.as_posix()}",
        f"- expected judge reply path: {judge_review_path.as_posix()}",
        "",
        "## Judge prompt",
        LLM_JUDGE_PROMPT_PATH.read_text(encoding="utf-8"),
        "",
        f"## Generation prompt ({PROMPT_PATH.as_posix()})",
        PROMPT_PATH.read_text(encoding="utf-8"),
    ]

    for label, path, text in optional_generation_inputs():
        sections += ["", f"## {label} ({path.as_posix()})", text]

    rules_text = read_rules_text(rules_path)
    if rules_text.strip():
        sections += ["", f"## Rule text ({rules_path.as_posix()})", rules_text]
    elif rules_path.suffix.lower() == ".pdf":
        page_paths = render_pdf_pages(rules_path)
        sections += [
            "",
            f"## Rulebook PDF ({rules_path.as_posix()})",
            "The PDF has no extractable text. Use these rendered page images as the rulebook source:",
        ]
        sections += [f"![{path.name}]({path.as_posix()})" for path in page_paths]
    else:
        sections += ["", "## Rule text", "_No extractable rule text found._"]

    sections += [
        "",
        f"## Generated code ({code_path.as_posix()})",
        "```python\n" + code_path.read_text(encoding="utf-8") + "```",
    ]

    if check_log_path.exists():
        sections += [
            "",
            f"## Check output ({check_log_path.as_posix()})",
            "```text\n" + check_log_path.read_text(encoding="utf-8") + "\n```",
        ]
    else:
        sections += [
            "",
            "## Check output",
            "_Run the checks cell first if you want to include deterministic check output._",
        ]

    output_path.write_text("\n\n".join(sections), encoding="utf-8")
    print(f"Saved judge packet: {output_path}")
    print(f"Save the judge reply as: {judge_review_path}")
    return output_path

SUMMARY_RE = re.compile(r"summary:\s+(\d+)/(\d+) checks, (\d+)/(\d+) units, (?:score=([0-9.]+), )?([0-9.]+)s")


def extract_summary(text: str) -> dict[str, int | float] | None:
    match = SUMMARY_RE.search(text)
    if match is None:
        return None
    return {
        "checks_passed": int(match.group(1)),
        "checks_total": int(match.group(2)),
        "units_passed": int(match.group(3)),
        "units_total": int(match.group(4)),
        "score": float(match.group(5)) if match.group(5) is not None else int(match.group(3)) / int(match.group(4)),
        "seconds": float(match.group(6)),
    }


print(f"Notebook ready for {RUN_VARIANT}: {CODE_PATH}")


## Agentic generation

This notebook keeps the original tool-using pi workflow:

- `pi -p`
- built-in tools only: `read,write,edit,bash,grep,find,ls`
- no extensions, skills, prompt templates, or context files
- pi reads the repo files itself and writes `outputs/<game>_agentic.py`
- the notebook also stores the raw final answer in `outputs/<game>_agentic.md`


In [ ]:
run_generation()


## Checks for the current variant

Run this after generation.

- both LLM-judge and OpenSpiel comparison toggles default to `True` for the full workflow
- keep `INCLUDE_LLM_JUDGE = True` after you saved the manual judge reply
- keep `INCLUDE_OPENSPIEL_COMPARE = True` for the optional OpenSpiel comparison


In [ ]:
INCLUDE_LLM_JUDGE = True
INCLUDE_OPENSPIEL_COMPARE = True

run_checks_for(
    code_path=CODE_PATH,
    judge_path=JUDGE_REVIEW_PATH,
    include_judge=INCLUDE_LLM_JUDGE,
    include_final=INCLUDE_OPENSPIEL_COMPARE,
    log_path=CHECK_LOG_PATH,
    label=RUN_VARIANT,
)


## Manual LLM judge preparation

Run the checks cell first. Then this cell writes a ready-to-paste judge packet with:

- the judge prompt
- the generation prompt and optional extra context
- the rule text
- the generated code
- the latest deterministic check output

Use that packet with your chosen judge model and save the reply to the printed `..._judge.md` path.


In [ ]:
build_judge_packet_for(
    code_path=CODE_PATH,
    check_log_path=CHECK_LOG_PATH,
    output_path=JUDGE_PACKET_PATH,
    judge_review_path=JUDGE_REVIEW_PATH,
    variant=RUN_VARIANT,
)


## Run both variants with the same checks

Use this after both `outputs/<game>_oneshot.py` and `outputs/<game>_agentic.py` exist.

- LLM-judge and OpenSpiel pair toggles default to `True` for the full side-by-side workflow
- keep the judge toggle on after both judge replies were saved
- keep the OpenSpiel toggle on for the optional reference comparison for both variants
- the pair action-language comparison normalizes emitted action names only; it does not add missing actions


In [ ]:

PAIR_INCLUDE_LLM_JUDGE = True
PAIR_INCLUDE_OPENSPIEL_COMPARE = True
PAIR_COMPARE_ACTION_LANGUAGE = True

pair_results = {}
for variant in ["oneshot", "agentic"]:
    paths = variant_paths(variant)
    if not Path(paths["code"]).exists():
        print(f"Missing {variant} code: {paths['code']}")
        continue

    print("\n" + "=" * 80)
    result = run_checks_for(
        code_path=Path(paths["code"]),
        judge_path=Path(paths["judge_review"]),
        include_judge=PAIR_INCLUDE_LLM_JUDGE,
        include_final=PAIR_INCLUDE_OPENSPIEL_COMPARE,
        log_path=Path(paths["check_log"]),
        label=variant,
    )
    pair_results[variant] = {
        "returncode": result.returncode,
        "summary": extract_summary((result.stdout or "") + (result.stderr or "")),
    }

if PAIR_COMPARE_ACTION_LANGUAGE and all(Path(variant_paths(variant)["code"]).exists() for variant in ["oneshot", "agentic"]):
    pair_compare_result = run_pair_action_compare(
        left_code_path=Path(variant_paths("oneshot")["code"]),
        right_code_path=Path(variant_paths("agentic")["code"]),
        left_label="oneshot",
        right_label="agentic",
    )
    pair_results["pair_action_compare"] = {
        "returncode": pair_compare_result.returncode,
        "summary": extract_summary((pair_compare_result.stdout or "") + (pair_compare_result.stderr or "")),
    }

print("\npair summary:")
for variant in ["oneshot", "agentic", "pair_action_compare"]:
    info = pair_results.get(variant)
    if info is None:
        print(f"- {variant}: not run")
        continue

    summary = info["summary"]
    if summary is None:
        print(f"- {variant}: returncode={info['returncode']} (no summary parsed)")
        continue

    print(
        f"- {variant}: {summary['checks_passed']}/{summary['checks_total']} checks, "
        f"{summary['units_passed']}/{summary['units_total']} units, "
        f"{summary['seconds']:.2f}s, returncode={info['returncode']}"
    )

if not all(variant in pair_results for variant in ["oneshot", "agentic"]):
    print("\nGenerate both variants first for a real side-by-side comparison.")
